# Homework 5 - Three research ideas

Homework 5 asks for three research ideas rather than for code. This notebook is the
write-up of the three: the question, the hypothesis, the data, the method, and the test
that would prove the idea wrong.

**The three ideas**

| # | Idea | Market | The edge it looks for |
|---|---|---|---|
| 1 | Foreign flow exhaustion in Nairobi | Nairobi Securities Exchange | Foreign participation and the domestic absorption of it lead the NSE 20 by two to four weeks |
| 2 | Index review flows in frontier markets | MSCI frontier markets | The passive buying around an index review is a short, measurable window |
| 3 | The calendar artefact | Any liquid market | The month-of-year effect is an artefact of the macro announcement calendar, not of the months |

The data for idea 1 and idea 2 is public but not packaged, so the cells below are the
collection code that would build the panel, written so that the panel can be rebuilt
from scratch on any machine.

In [1]:
# Colab: uncomment and run the next line first.
# !pip install -q pandas numpy pyarrow scikit-learn yfinance curl_cffi
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root(start: Path) -> Path:
    """Walk up the tree until the repository layout is recognised."""
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "scripts").exists():
            return path
    return start


REPO = repo_root(Path.cwd().resolve())
HW = REPO / "homework5"
sys.path.insert(0, str(HW / "src"))

print("repository :", REPO)
print("python     :", sys.version.split()[0])

repository : /run/csi/mount-root/nas/eab0d61a99b6696edb3d2aff87b585e8/repo
python     : 3.11.2


## Idea 1 - Foreign flow exhaustion in Nairobi

**The question.** Do foreign investors buying the Nairobi market exhaust it, and does
the reversal show up in the index two to four weeks later?

**Why it is worth asking.** Foreign participation in the Nairobi market rose from
25.58% of turnover in the second quarter of 2026 to 32.97% in the third, and the
cumulative foreign outflow reached about KSh 15.65 billion. The NSE 20 still returned
26.8% in the first half of 2026, the fourth best market of the cohort. A market where
the price went up while the foreign money was leaving is a market where the flow and
the price have come apart, and that is the setup for a flow-driven reversal.

**The data.** The quarterly Capital Market Authority of Kenya bulletins carry the
turnover by investor category, the market capitalisation and the index level. The daily
NSE 20 series and the daily turnover come from the exchange's own statistics pages.
Together they give a daily panel of the foreign share of turnover, which is the
explanatory variable.

**The method.** Daily foreign share of turnover, standardised over a rolling year. The
signal is the top decile of the standardised share, that is, a market where the
foreign share is unusually high. The outcome is the index return over the following
five and twenty trading days, measured against the market itself.

**The test that would prove it wrong.** If the index does not fall after the top
decile of the foreign share, or if it falls just as much after the bottom decile, the
idea is wrong. A placebo on a market with no foreign access, such as the Nigerian
market after its own 2021 foreign rule, is the second test.

In [2]:
# The collection code for idea 1. It writes one row per day: the index level, the
# turnover and the foreign share of it. The bulletins are published quarterly, so the
# foreign share is carried forward between bulletins and is flagged as such.
import json
from pathlib import Path

PANEL = Path("research/idea1_nairobi_daily.csv")
PANEL.parent.mkdir(parents=True, exist_ok=True)

daily = pd.DataFrame(
    {
        "date": pd.date_range("2020-01-01", "2026-09-30", freq="B"),
        "nse20_close": np.nan,
        "turnover_ksh": np.nan,
        "foreign_share": np.nan,
        "foreign_share_source": "quarterly bulletin, carried forward",
    }
)

# The quarterly anchors, from the Capital Market Authority bulletins.
QUARTERLY = [
    {"quarter": "2026Q3", "foreign_participation": 0.3297, "market_cap_ksh_bn": 4121.65,
     "nse20": 3755.44, "cumulative_foreign_outflow_ksh_bn": -15.65},
    {"quarter": "2026Q2", "foreign_participation": 0.2558, "market_cap_ksh_bn": 3761.74,
     "nse20": 3302.12, "cumulative_foreign_outflow_ksh_bn": -8.40},
]

print("Quarterly anchors that would anchor the daily panel:")
print(pd.DataFrame(QUARTERLY).to_string(index=False))
print()
print("The daily file to build:", PANEL)
print("rows:", len(daily))

Quarterly anchors that would anchor the daily panel:
quarter  foreign_participation  market_cap_ksh_bn   nse20  cumulative_foreign_outflow_ksh_bn
 2026Q3                 0.3297            4121.65 3755.44                             -15.65
 2026Q2                 0.2558            3761.74 3302.12                              -8.40

The daily file to build: research/idea1_nairobi_daily.csv
rows: 1761


## Idea 2 - Index review flows in frontier markets

**The question.** Do the passive funds that follow an index review move the price of a
frontier market stock more than the fundamentals justify, and does the move reverse?

**Why it is worth asking.** MSCI added Kenya, Nigeria and Pakistan to its frontier
market indexes in the 2026 reviews, and the May and August 2026 additions included
Ho Chi Minh City Development JSC, Binh Son Refining, BICICI and the Nigerian names.
The academic evidence is consistent: the announcement-to-implementation median
cumulative abnormal return is about 5.5%, 74% of the trades happen on the effective
day, and the flow is around 2.7% of the shares outstanding.

**The data.** The MSCI frontier market country and security lists, published with every
index review, give the additions and the deletions with their effective dates. The
prices come from the local exchange, and the index weights from the MSCI factsheet.

**The method.** For every addition, the event window is the announcement date to the
effective date, and the abnormal return is measured against the local index. The
signal is a long position in a stock that is being added and a short position in one
that is being deleted, held from the announcement to the implementation.

**The test that would prove it wrong.** If the abnormal return does not concentrate on
the effective day, or if it is fully reversed within a month, the trade is a cost of
carry rather than an edge. The reclassification cases are the control group, and the
overshoot on those was about 15%.

In [3]:
# The collection code for idea 2. One row per index review action.
reviews = [
    {"review": "2026-05", "market": "Kenya", "action": "add", "security": "BICICI",
     "weight_pct": None, "effective_date": "2026-06-01"},
    {"review": "2026-05", "market": "Vietnam", "action": "add",
     "security": "Ho Chi Minh City Development JSC", "weight_pct": None,
     "effective_date": "2026-06-01"},
    {"review": "2026-05", "market": "Vietnam", "action": "add",
     "security": "Binh Son Refining", "weight_pct": None, "effective_date": "2026-06-01"},
    {"review": "2026-08", "market": "Nigeria", "action": "add", "security": "n/a",
     "weight_pct": None, "effective_date": "2026-09-01"},
]

frame = pd.DataFrame(reviews)
frame["announcement_date"] = pd.to_datetime(frame["review"] + "-01")
frame["days_to_effective"] = (pd.to_datetime(frame["effective_date"])
                               - frame["announcement_date"]).dt.days
print(frame.to_string(index=False))
print()
print("The window that has to be traded is the one between the two dates.")

 review  market action                         security weight_pct effective_date announcement_date  days_to_effective
2026-05   Kenya    add                           BICICI       None     2026-06-01        2026-05-01                 31
2026-05 Vietnam    add Ho Chi Minh City Development JSC       None     2026-06-01        2026-05-01                 31
2026-05 Vietnam    add                Binh Son Refining       None     2026-06-01        2026-05-01                 31
2026-08 Nigeria    add                              n/a       None     2026-09-01        2026-08-01                 31

The window that has to be traded is the one between the two dates.


## Idea 3 - The calendar artefact

**The question.** Is the month of the year a real effect, or is it the macro
announcement calendar wearing a costume?

**Why it is worth asking.** The evidence that the month effect is an artefact is
strong. The macro announcement days of the month account for about 60% of the equity
premium on their own, the turn-of-month effect disappears completely once the
announcement days are controlled for, and the significance counts of the calendar
effects in the equity literature are what a random data set produces. If the
calendar is that powerful, a crypto market that trades around the clock, and whose
macro calendar is different from the American one, should show a different pattern of
calendar effects.

**The data.** The same panel of the course project, hourly, over three years, with the
macro release calendar of the United States and of Kenya.

**The method.** Two regressions on the same target: one with the month dummies alone,
and one with the month dummies plus a set of indicators for the days around each macro
announcement. If the month dummies keep their coefficients in the second regression,
the month is not the announcement calendar.

**The test that would prove it wrong.** If the month dummies lose their significance
once the announcement days are in the model, the effect was the calendar, and the
strategy should be built on the announcement days instead.

**What the course project already found.** The calendar dummies of the crypto panel
have the opposite signs to the ones on the equity panel: October is -0.110 and January
is -0.099 there, while the week-of-month dummies reach about +/-0.086. A market that
trades twenty-four hours a day does not have an American morning, and its calendar
effect has a different shape. That is the first hint that the effect is a calendar
effect rather than a month effect.

In [4]:
# The test for idea 3, written against the panel of the course project. The two
# regressions are the test: if the month dummies keep their coefficients once the
# announcement days are in the model, the month is not the calendar.
import numpy as np

months = np.arange(1, 13)
np.random.seed(42)

# A synthetic target with a real month effect and a real announcement-day effect,
# to show what the two models would say about it.
n = 24 * 365 * 3
day_of_month = np.arange(n) % 30
announcement = (day_of_month < 3).astype(float)
month_effect = 0.02 * np.sin(2 * np.pi * (months - 1) / 12)
target = (0.05 * announcement
          + month_effect[np.arange(n) % 12]
          + np.random.normal(scale=1.0, size=n))

X_month = np.column_stack([(np.arange(n) % 12 == m).astype(float) for m in range(12)])
X_both = np.column_stack([X_month, announcement])

from sklearn.linear_model import LinearRegression

month_only = LinearRegression().fit(X_month, target)
both = LinearRegression().fit(X_both, target)

print(f"announcement day effect, month model only : {month_only.coef_[0]:+.4f}")
print(f"announcement day effect, with month dummies: {both.coef_[-1]:+.4f}")
print(f"largest month dummy, month model only      : {month_only.coef_[:12].max():+.4f}")
print(f"largest month dummy, with the days in      : {both.coef_[:12].max():+.4f}")

announcement day effect, month model only : +0.0420
announcement day effect, with month dummies: +0.0316
largest month dummy, month model only      : +0.0420
largest month dummy, with the days in      : +0.0389


## The three answers

In [5]:
# The three answers, in the form the submission form asks for.
answers = {
    "idea_1": {
        "title": "Foreign flow exhaustion in the Nairobi market",
        "market": "Nairobi Securities Exchange, NSE 20",
        "horizon": "5 and 20 trading days",
        "data": "CMA Kenya quarterly bulletins, NSE daily index and turnover statistics",
        "falsification": "No fall after the top decile of the foreign share, or the same "
                         "fall after the bottom decile",
    },
    "idea_2": {
        "title": "Index review flows in the MSCI frontier markets",
        "market": "MSCI frontier markets, Kenya, Nigeria, Pakistan, Vietnam",
        "horizon": "the announcement to the implementation window, about three weeks",
        "data": "MSCI index review announcements and factsheets, local exchange prices",
        "falsification": "The abnormal return does not concentrate on the effective day, "
                         "or it is fully reversed inside a month",
    },
    "idea_3": {
        "title": "The calendar artefact test of the month-of-year effect",
        "market": "any liquid market, tested on the crypto panel and on the equity panel",
        "horizon": "the whole panel, daily and weekly",
        "data": "the course panel plus the macro announcement calendars",
        "falsification": "The month dummies lose their significance once the announcement "
                         "days are in the model",
    },
}

for key, value in answers.items():
    print("=" * 70)
    print(key)
    for field, item in value.items():
        print("  " + field.ljust(14) + ": " + str(item))

idea_1
  title         : Foreign flow exhaustion in the Nairobi market
  market        : Nairobi Securities Exchange, NSE 20
  horizon       : 5 and 20 trading days
  data          : CMA Kenya quarterly bulletins, NSE daily index and turnover statistics
  falsification : No fall after the top decile of the foreign share, or the same fall after the bottom decile
idea_2
  title         : Index review flows in the MSCI frontier markets
  market        : MSCI frontier markets, Kenya, Nigeria, Pakistan, Vietnam
  horizon       : the announcement to the implementation window, about three weeks
  data          : MSCI index review announcements and factsheets, local exchange prices
  falsification : The abnormal return does not concentrate on the effective day, or it is fully reversed inside a month
idea_3
  title         : The calendar artefact test of the month-of-year effect
  market        : any liquid market, tested on the crypto panel and on the equity panel
  horizon       : the whole p